In [1]:
import os
import sys
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import openpyxl
from peft import PeftModel
import pandas as pd

sys.path.append("D:\\Dissertacao\\slm_tupi")
from utils.utils_train import create_prompt

sys.path.append("D:\\Dissertacao\\slm_tupi\\experiments")
from tupi_exp.config import SUBCONJUNTO, seed_of_run

c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths: run_dir is the output_dir of the SFT run (checkpoints + the test split written by that same run)
base_model = "Qwen/Qwen2.5-3B-Instruct"
run_dir = r"D:\Dissertacao\slm_tupi\experiments\exp01_hparams\logs\exp01_hparams\peft_64_Tu_Po\fit_10_31_23_03_1.0_dataset_sintetico_aumentado-SEED42"
run_seed = seed_of_run(run_dir)  # from the -SEED<N> suffix of the run folder
lora_path = os.path.join(run_dir, "checkpoint-516")
data_path = os.path.join(run_dir, "test_dataset.json")
use_lora = True  # False: zero-shot baseline (base model without the adapter), same prompt and test set

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    trust_remote_code=True
)

# Base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    torch_dtype="auto",
    device_map="auto"
)

# Load LoRA
if use_lora:
    model = PeftModel.from_pretrained(
        model,
        lora_path
    )

model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [3]:
# Load Dataset
df = pd.read_json(
    data_path,
    lines=True
)

# Prompt and Inference
predictions = []

for idx, row in df.iterrows():

    tupi_text = row["Tupi"]

    # Same chat template as in training (ends with "<|im_start|>assistant\n")
    prompt = create_prompt(row, src_lng="Tupi", tgt_lng="Português", mode="inference", tokenizer=tokenizer)["full_prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,  # greedy: deterministic
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens, not the prompt
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    predictions.append({
        "Tupi": tupi_text,
        "fonte": row["fonte"],
        "Português_esperado": row["Referência"],
        "Português_gemini": row["Português"] if row["fonte"] == "gemini" else None,
        "Português_predito": response,
        "Subconjunto": SUBCONJUNTO[row["fonte"]],
    })

    print("=" * 80)
    print("TUPI:")
    print(tupi_text)

    print("\nESPERADO:")
    print(row["Referência"])

    print("\nPREDITO:")
    print(response)

c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(

TUPI:
Taçó ndé irùmo Sor guí.

ESPERADO:
- Hei de ir contigo, ó senhor.

PREDITO:
Certamente tu vives no Taça.
TUPI:
abá ogûatá xe irunamo oka rupi taperá suí osykyîébo

ESPERADO:
o homem caminhou comigo pela casa tendo medo da andorinha

PREDITO:
o homem caminhou comigo pela casa tendo medo da andorinha
TUPI:
Coritéi i có ára oçaçáo,

ESPERADO:
Logo este mundo passa,

PREDITO:
Fiquei lá bem no dia da morte dele,
TUPI:
Cuias oguerúr moçapýr,

ESPERADO:
e trouxe três cuias.

PREDITO:
Dei-lhe muitas coisas,
TUPI:
agûaragûasu nhu reséndûara oîabab

ESPERADO:
o lobo-guará do campo foge

PREDITO:
o lobo-guará do campo foge
TUPI:
abá ogûatá xe irunamo ybytýra rupi tatu suí osykyîébo

ESPERADO:
o homem caminhou comigo pela serra tendo medo do tatu

PREDITO:
o homem caminhou comigo pela serra tendo medo do tatu
TUPI:
îagûara oîpysýkyba'epûera xe ruba

ESPERADO:
quem capturou a onça foi meu pai

PREDITO:
quem capturou a onça foi meu mesmo
TUPI:
abá ogûatá xe irunamo taba rupi gûyrá suí osykyîéb

In [4]:
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    os.path.join(run_dir, f"predictions_exp01_hparams-SEED{run_seed}.csv" if use_lora else f"predictions_base-SEED{run_seed}.csv"),
    index=False
)